In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import json
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')


In [3]:
# (CHANGE THESE ACCORDING TO YOUR DRIVE)
STUDENTLIFE_PATH = "/content/drive/MyDrive/AI Project - Academic Burnout/Data"
#OUTPUT_PATH = "/content/drive/MyDrive/AI_Burnout_Predictor/results_realistic_studentlife"

#os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Paths configured.")


Paths configured.


In [4]:
# Loading StudentLife data
def load_studentlife_json(folder_path):
    all_data = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".json"):
            student_id = file_name.replace(".json", "")
            with open(os.path.join(folder_path, file_name), "r") as f:
                records = json.load(f)
                for r in records:
                    r["student_id"] = student_id
                    all_data.append(r)
    return pd.DataFrame(all_data)

print("Loading StudentLife...")

stress_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Stress"))
activity_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Activity"))
sleep_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Sleep"))

print(f"Stress: {stress_raw.shape}")
print(f"Activity: {activity_raw.shape}")
print(f"Sleep: {sleep_raw.shape}")


Loading StudentLife...
Stress: (2408, 5)
Activity: (833, 9)
Sleep: (1644, 7)


In [5]:
stress_raw.head()

,level,location,resp_time,student_id,null
0,3,"43.70712148268646,-72.29087061546424",1364681195,Stress_u49,NaN
1,NaN,NaN,1364121911,Stress_u49,1
2,NaN,NaN,1364121426,Stress_u49,1
3,NaN,NaN,1364121429,Stress_u49,1
4,NaN,NaN,1364118814,Stress_u49,"43.70633605,-72.28701846"


In [6]:
activity_raw.head()

,Social2,null,resp_time,student_id,other_relaxing,other_working,relaxing,working,location
0,2,3,1364853391,Activity_u12,NaN,NaN,NaN,NaN,NaN
1,3,4,1364359262,Activity_u12,NaN,NaN,NaN,NaN,NaN
2,2,1,1364592822,Activity_u12,NaN,NaN,NaN,NaN,NaN
3,3,1,1364420311,Activity_u12,NaN,NaN,NaN,NaN,NaN
4,3,1,1364537832,Activity_u12,NaN,NaN,NaN,NaN,NaN


In [7]:
sleep_raw.head()

,null,resp_time,student_id,hour,location,rate,social
0,"43.70643224,-72.28718618",1364121462,Sleep_u25,NaN,NaN,NaN,NaN
1,7,1364118945,Sleep_u25,NaN,NaN,NaN,NaN
2,"43.70643224,-72.28718618",1364121461,Sleep_u25,NaN,NaN,NaN,NaN
3,2,1364121789,Sleep_u25,NaN,NaN,NaN,NaN
4,"43.70643224,-72.28718618",1364121464,Sleep_u25,NaN,NaN,NaN,NaN


In [8]:
# Cleaning the STRESS DATASET
# =========================
print("\n DATASET: STRESS")
print("Shows self-reported student stress levels over time")

print("\nCleaning StudentLife Stress...")

stress_clean = stress_raw.copy()

# Dropping the 'null' column
if 'null' in stress_clean.columns:
    stress_clean = stress_clean.drop(columns=['null'])

print("\nInitial Stress Dataset:")
print(stress_clean)

# Converting  timestamp
stress_clean['timestamp'] = pd.to_datetime(stress_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(stress_clean)

# Cleaning  student_id
stress_clean['student_id'] = stress_clean['student_id'].str.replace('Stress_', '')
print("\nAfter cleaning student_id:")
print(stress_clean)



 DATASET: STRESS
Shows self-reported student stress levels over time

Cleaning StudentLife Stress...

Initial Stress Dataset:
     level                              location   resp_time  student_id
0        3  43.70712148268646,-72.29087061546424  1364681195  Stress_u49
1      NaN                                   NaN  1364121911  Stress_u49
2      NaN                                   NaN  1364121426  Stress_u49
3      NaN                                   NaN  1364121429  Stress_u49
4      NaN                                   NaN  1364118814  Stress_u49
...    ...                                   ...         ...         ...
2403     2              43.70390811,-72.29065789  1369368560  Stress_u16
2404     2              43.70675104,-72.28740962  1369349334  Stress_u16
2405     1              43.70372826,-72.29097234  1369428881  Stress_u16
2406     3              43.68753666,-72.29250241  1369719704  Stress_u16
2407     3              43.68753666,-72.29250241  1369729103  Stress_u

In [9]:
#renaming the level column
stress_clean = stress_clean.rename(columns={'level': 'stress_level'})
print("\nAfter renaming level → stress_level:")
print(stress_clean)

#Converting stress_level to numeric data
stress_clean['stress_level'] = pd.to_numeric(stress_clean['stress_level'], errors='coerce')
print("\nAfter converting stress_level to numeric:")
print(stress_clean)

if 'null' in stress_clean.columns:

    # if stress_level is missing but null looks like numeric, then use this
    null_as_num = pd.to_numeric(stress_clean['null'], errors='coerce')
    fill_mask = stress_clean['stress_level'].isna() & null_as_num.notna()
    if fill_mask.any():
        stress_clean.loc[fill_mask, 'stress_level'] = null_as_num.loc[fill_mask]

    #If location is missing but null looks like "lat,long", then use this
    if 'location' in stress_clean.columns:
        null_as_str = stress_clean['null'].astype(str)
        coord_mask = stress_clean['location'].isna() & null_as_str.str.match(
            r'^-?\d+(\.\d+)?,-?\d+(\.\d+)?$'
        )
        if coord_mask.any():
            stress_clean.loc[coord_mask, 'location'] = stress_clean.loc[coord_mask, 'null']

print("\nAfter recovering values from 'null' (if applicable):")
print(stress_clean)

#dropping missing stress values
stress_clean = stress_clean.dropna(subset=['stress_level'])
print("\nAfter dropping NaN stress levels:")
print(stress_clean)

#Explicit float conversion
stress_clean['stress_level'] = stress_clean['stress_level'].astype(float)
print("\nFinal cleaned Stress dataset:")
print(stress_clean)



After renaming level → stress_level:
     stress_level                              location   resp_time  \
0               3  43.70712148268646,-72.29087061546424  1364681195   
1             NaN                                   NaN  1364121911   
2             NaN                                   NaN  1364121426   
3             NaN                                   NaN  1364121429   
4             NaN                                   NaN  1364118814   
...           ...                                   ...         ...   
2403            2              43.70390811,-72.29065789  1369368560   
2404            2              43.70675104,-72.28740962  1369349334   
2405            1              43.70372826,-72.29097234  1369428881   
2406            3              43.68753666,-72.29250241  1369719704   
2407            3              43.68753666,-72.29250241  1369729103   

     student_id           timestamp  
0           u49 2013-03-30 22:06:35  
1           u49 2013-03-24 10:45:

In [10]:
# Code for Activity data cleaning
print("\n DATASET: ACTIVITY")
print(" Shows students' daily activity levels (social, working, relaxing)")

print("\nCleaning StudentLife Activity")

activity_clean = activity_raw.copy()

# Drop the 'null' coloumns
if 'null' in activity_clean.columns:
    activity_clean = activity_clean.drop(columns=['null'])

print("\nInitial Activity Dataset:")
print(activity_clean)

# Convert timestamp

activity_clean['timestamp'] = pd.to_datetime(activity_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(activity_clean)

# Clean student_id
activity_clean['student_id'] = activity_clean['student_id'].str.replace('Activity_', '')
print("\nAfter cleaning student_id:")
print(activity_clean)



 DATASET: ACTIVITY
 Shows students' daily activity levels (social, working, relaxing)

Cleaning StudentLife Activity

Initial Activity Dataset:
    Social2   resp_time    student_id other_relaxing other_working relaxing  \
0         2  1364853391  Activity_u12            NaN           NaN      NaN   
1         3  1364359262  Activity_u12            NaN           NaN      NaN   
2         2  1364592822  Activity_u12            NaN           NaN      NaN   
3         3  1364420311  Activity_u12            NaN           NaN      NaN   
4         3  1364537832  Activity_u12            NaN           NaN      NaN   
..      ...         ...           ...            ...           ...      ...   
828     NaN  1365754502  Activity_u30              3             3        1   
829     NaN  1366495759  Activity_u30              2             1        1   
830     NaN  1366575978  Activity_u30              2             2        1   
831     NaN  1367258363  Activity_u30              2             

In [11]:


# key columns of studentlife dataset

for col in ["Social2", "working", "other_working", "relaxing", "other_relaxing"]:
    if col not in activity_clean.columns:
        activity_clean[col] = np.nan



# Converting to numeric

for col in ["Social2", "working", "other_working", "relaxing", "other_relaxing"]:
  activity_clean[col] = pd.to_numeric(activity_clean[col], errors="coerce")



# Computing interpretable scores

activity_clean["workload_score"] = activity_clean[["working", "other_working"]].sum(axis=1, min_count=1)
activity_clean["recovery_score"] = activity_clean[["relaxing", "other_relaxing"]].sum(axis=1, min_count=1)
activity_clean["social_score"] = activity_clean["Social2"]


print("\nAfter computing workload_score / recovery_score / social_score:")

print(activity_clean[["student_id", "timestamp", "workload_score", "recovery_score", "social_score"]].head())



# Keep relevant columns and drop rows where ALL scores are missing

activity_clean = activity_clean[["student_id", "timestamp", "workload_score", "recovery_score", "social_score"]]
activity_clean = activity_clean.dropna( subset=["workload_score", "recovery_score", "social_score"], how="all" ).copy()



print("\nFinal cleaned Activity dataset (3 scores):")
print(activity_clean.head())




After computing workload_score / recovery_score / social_score:
  student_id           timestamp  workload_score  recovery_score  social_score
0        u12 2013-04-01 21:56:31             NaN             NaN           2.0
1        u12 2013-03-27 04:41:02             NaN             NaN           3.0
2        u12 2013-03-29 21:33:42             NaN             NaN           2.0
3        u12 2013-03-27 21:38:31             NaN             NaN           3.0
4        u12 2013-03-29 06:17:12             NaN             NaN           3.0

Final cleaned Activity dataset (3 scores):
  student_id           timestamp  workload_score  recovery_score  social_score
0        u12 2013-04-01 21:56:31             NaN             NaN           2.0
1        u12 2013-03-27 04:41:02             NaN             NaN           3.0
2        u12 2013-03-29 21:33:42             NaN             NaN           2.0
3        u12 2013-03-27 21:38:31             NaN             NaN           3.0
4        u12 2013-03-2

In [12]:
# Sleep dataset processing

# Print dataset name
print("\ DATASET: SLEEP")

# Describe dataset purpose
print("Shows students' self-reported sleep duration in hours")

# Start cleaning process
print("\nCleaning StudentLife Sleep...")

# Create a copy of raw dataset
sleep_clean = sleep_raw.copy()

# Remove irrelevant null column if present
if 'null' in sleep_clean.columns:
    sleep_clean = sleep_clean.drop(columns=['null'])

# Display initial dataset
print("\nInitial Sleep Dataset:")
print(sleep_clean)

# Convert response time to timestamp
sleep_clean['timestamp'] = pd.to_datetime(sleep_clean['resp_time'], unit='s')

# Show dataset after timestamp conversion
print("\nAfter converting resp_time to timestamp:")
print(sleep_clean)

\ DATASET: SLEEP
Shows students' self-reported sleep duration in hours

Cleaning StudentLife Sleep...

Initial Sleep Dataset:
       resp_time student_id hour                  location rate social
0     1364121462  Sleep_u25  NaN                       NaN  NaN    NaN
1     1364118945  Sleep_u25  NaN                       NaN  NaN    NaN
2     1364121461  Sleep_u25  NaN                       NaN  NaN    NaN
3     1364121789  Sleep_u25  NaN                       NaN  NaN    NaN
4     1364121464  Sleep_u25  NaN                       NaN  NaN    NaN
...          ...        ...  ...                       ...  ...    ...
1639  1365962124  Sleep_u23    8   43.7050633,-72.28352333    1      1
1640  1366047881  Sleep_u23    8  43.70548232,-72.28294733    1      1
1641  1366220438  Sleep_u23    6  43.70563203,-72.28296566    1      1
1642  1366491749  Sleep_u23    8  43.70587021,-72.28397539    1      1
1643  1366650172  Sleep_u23    6  43.70500519,-72.28332888    1      1

[1644 rows x 6 column

In [13]:
# Clean student_id by removing prefix
sleep_clean['student_id'] = sleep_clean['student_id'].str.replace('Sleep_', '')

# Show dataset after cleaning student_id
print("\nAfter cleaning student_id:")
print(sleep_clean)

# Convert hour column to numeric sleep_hours
sleep_clean['sleep_hours'] = pd.to_numeric(sleep_clean['hour'], errors='coerce')

# Show dataset after converting sleep hours
print("\nAfter converting hour to sleep_hours:")
print(sleep_clean)

# Recover missing sleep hours from null column if available
if 'null' in sleep_clean.columns:
    null_as_num = pd.to_numeric(sleep_clean['null'], errors='coerce')
    fill_mask = sleep_clean['sleep_hours'].isna() & null_as_num.notna()
    if fill_mask.any():
        sleep_clean.loc[fill_mask, 'sleep_hours'] = null_as_num.loc[fill_mask]

# Show dataset after recovery step
print("\nAfter recovering sleep_hours from null if applicable:")
print(sleep_clean)

# Keep only relevant columns and remove missing values
sleep_clean = sleep_clean[['student_id', 'timestamp', 'sleep_hours']].dropna()

# Display final cleaned dataset
print("\nFinal cleaned Sleep dataset:")
print(sleep_clean)


After cleaning student_id:
       resp_time student_id hour                  location rate social  \
0     1364121462        u25  NaN                       NaN  NaN    NaN   
1     1364118945        u25  NaN                       NaN  NaN    NaN   
2     1364121461        u25  NaN                       NaN  NaN    NaN   
3     1364121789        u25  NaN                       NaN  NaN    NaN   
4     1364121464        u25  NaN                       NaN  NaN    NaN   
...          ...        ...  ...                       ...  ...    ...   
1639  1365962124        u23    8   43.7050633,-72.28352333    1      1   
1640  1366047881        u23    8  43.70548232,-72.28294733    1      1   
1641  1366220438        u23    6  43.70563203,-72.28296566    1      1   
1642  1366491749        u23    8  43.70587021,-72.28397539    1      1   
1643  1366650172        u23    6  43.70500519,-72.28332888    1      1   

               timestamp  
0    2013-03-24 10:37:42  
1    2013-03-24 09:55:45  
2 

In [14]:
# Sahitya_week2 added code for feature engineering for stress dataset
print("CREATING WEEKLY STUDENTLIFE FEATURES (STRESS + SLEEP + ACTIVITY)")

for df, time_col in [  # loop through each dataframe and its timestamp column name

    (stress_clean, "timestamp"),  # pair: stress dataframe + timestamp column

    (activity_clean, "timestamp"),  # pair: activity dataframe + timestamp column

    (sleep_clean, "timestamp"),  # pair: sleep dataframe + timestamp column

]:

    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")  # convert to datetime;

stress_clean["stress_level"] = pd.to_numeric(stress_clean["stress_level"], errors="coerce")  # convert stress to numeric;

for col in ["workload_score","recovery_score","social_score"]:  # iterate through activity score columns

    if col in activity_clean.columns:  # check if the column exists in activity data

        activity_clean[col] = pd.to_numeric(activity_clean[col], errors="coerce")  # convert that column to numeric;

sleep_clean["sleep_hours"] = pd.to_numeric(sleep_clean["sleep_hours"], errors="coerce")  # convert sleep hours to numeric;

sleep_clean.head()

CREATING WEEKLY STUDENTLIFE FEATURES (STRESS + SLEEP + ACTIVITY)


,student_id,timestamp,sleep_hours
5,u25,2013-03-30 18:15:44,9.0
6,u25,2013-03-28 17:54:22,8.0
7,u25,2013-04-02 17:20:27,8.0
8,u25,2013-04-01 05:27:26,8.0
9,u25,2013-04-01 04:11:26,7.0


the following code is written by jaimil kothari - week 2

In [16]:


def add_student_zscore(df, id_col, value_col, new_col, time_col=None):  # to define helper function to add a z-score column

    if time_col is None:  # checking if a time column name was provided or not
        # falling back to index order if timestamp isn't provided
        time_col = None  # keeping it none explicitly

    if time_col is not None and time_col in df.columns:  # if time_col exists then doing sort by id + time
        df = df.sort_values([id_col, time_col]).copy()  # sorting data to ensure expanding stats follow time order

    else:  # else sort only by student id
        df = df.sort_values([id_col]).copy()  # sorting by student id for deterministic order

    def _exp_z(s: pd.Series) -> pd.Series:  # this is an inner function to compute expanding z-score for one student series
        exp_mean = s.expanding(min_periods=1).mean()  # expanding mean up to each point
        exp_std = s.expanding(min_periods=2).std().fillna(0.0)  # expanding std; needs >=2 points
        exp_std = exp_std.replace(0, np.nan).fillna(1.0)  # replacing 0 std with 1 to avoid divide-by-zero
        return (s - exp_mean) / exp_std  # computing z-score using expanding mean/std


    df[new_col] = df.groupby(id_col)[value_col].transform(_exp_z)  # expanding z-score per student
    return df  # returning dataframe with new z-score column added


stress_clean = add_student_zscore(stress_clean, "student_id", "stress_level", "stress_z", time_col="timestamp")  # adding stress_z

for col, zcol in [("workload_score","workload_z"),("recovery_score","recovery_z"),("social_score","social_z")]:  # mapping activity cols to z cols
    if col in activity_clean.columns:  # true only if the activity column exists
        activity_clean = add_student_zscore(activity_clean, "student_id", col, zcol, time_col="timestamp")  # adding corresponding z-score column

sleep_clean = add_student_zscore(sleep_clean, "student_id", "sleep_hours", "sleep_z")  # adding sleep_z (no time_col passed here)

In [17]:
stress_clean.head()

,stress_level,location,resp_time,student_id,timestamp,stress_z
2078,2.0,"43.70692415,-72.2873929",1364237696,u00,2013-03-25 18:54:56,0.000000
2079,2.0,"43.70555193,-72.28704778",1364268806,u00,2013-03-26 03:33:26,0.000000
2080,2.0,"43.70555193,-72.28704778",1364268814,u00,2013-03-26 03:33:34,0.000000
2081,1.0,"43.70678675,-72.28732051",1364346740,u00,2013-03-27 01:12:20,-1.500000
2083,1.0,"43.70508322,-72.28677496",1364437527,u00,2013-03-28 02:25:27,-1.095445


In [18]:
sleep_clean.head(100)

,student_id,timestamp,sleep_hours,sleep_z
145,u00,2013-05-28 17:54:35,9.0,0.000000
132,u00,2013-05-01 17:39:33,3.0,-0.707107
131,u00,2013-05-01 17:39:34,3.0,-0.577350
130,u00,2013-05-01 17:39:53,3.0,-0.500000
128,u00,2013-04-22 17:06:04,3.0,-0.447214
...,...,...,...,...
781,u02,2013-04-02 18:10:01,8.0,0.424212
789,u02,2013-04-13 17:03:11,8.0,0.408248
788,u02,2013-04-10 18:25:00,8.0,0.393974
787,u02,2013-04-09 17:00:33,7.0,-0.823329
